# Prepare Data

1. copy into BIDS format folder structure and rename
2. add necessary .json (for each .nii and 2 general ones in the root folder)
3. redo T1.nii.gz gzip (rename to nii - they were not proper -  and then gzip)


Organization:
pati = op.join(source_folder, group_name, 'XX', 'XX_..', 'XX_....nii.gz'); 
_3DT1
_fMRI_preT_
_fMRI_postT_


In [1]:
import numpy as np
import pandas as pd
import os
import os.path as op
import shutil
from glob import glob

bids_folder = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline'

In [ ]:
# --- USER INPUT ---
source_folder = '/mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/from_Karin/MRI_Data'
target_bids_root = bids_folder
groups = ['Dyscalculics', 'Controls']

# --- Helper functions ---
def make_bids_id(orig_id, group):
    """Format subject ID for BIDS"""
    if group == 'Dyscalculics':
        return f"sub-1{int(orig_id):02d}"  # 2-digit ID + prefix 1 = 3-digit
    else:
        return f"sub-{int(orig_id):03d}"

def safe_mkdir(path):
    os.makedirs(path, exist_ok=True)


In [7]:
# --- Main loop ---
for group in groups:
    group_path = os.path.join(source_folder, group)
    subjects = os.listdir(group_path)
    
    for subj in subjects:
        subj_path = os.path.join(group_path, subj)
        if not os.path.isdir(subj_path):
            continue
        
        scan_dirs = glob(os.path.join(subj_path, f"{subj}_*_*"))
        bids_id = make_bids_id(subj, group)

        for scan_dir in scan_dirs:
            scan_name = os.path.basename(scan_dir)
            if '3DT1' in scan_name:
                ses = '1'
                modality = 'anat'
                out_fname = f"{bids_id}_ses-{ses}_T1w.nii.gz"
            elif 'fMRI_preT' in scan_name:
                ses = '1'
                modality = 'func'
                out_fname = f"{bids_id}_ses-{ses}_task-digitorder_bold.nii.gz"
            elif 'fMRI_postT' in scan_name:
                ses = '2'
                modality = 'func'
                out_fname = f"{bids_id}_ses-{ses}_task-digitorder_bold.nii.gz"
            else:
                print(f"Unknown scan type in folder: {scan_name} — skipping")
                continue

            nii_files = glob(os.path.join(scan_dir, "*.nii*"))
            if len(nii_files) != 1:
                print(f"Warning: Found {len(nii_files)} NIfTI files in {scan_dir}, expected 1.")
                continue
            nii_file = nii_files[0]

            # Create BIDS output path
            dest_dir = os.path.join(target_bids_root, bids_id, f"ses-{ses}", modality)
            safe_mkdir(dest_dir)

            dest_path = os.path.join(dest_dir, out_fname)
            shutil.copy(nii_file, dest_path)
            print(f"Copied {nii_file} → {dest_path}")


Copied /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/from_Karin/MRI_Data/Controls/304/304_3DT1_FSPGR_172_Anonymised/304_3DT1.nii → /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-304/ses-1/anat/sub-304_ses-1_T1w.nii.gz
Copied /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/from_Karin/MRI_Data/Controls/304/304_fMRI_postT_Anonymised/304_fMRI_postT.nii.gz → /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-304/ses-2/func/sub-304_ses-2_task-digitorder_bold.nii.gz
Copied /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/from_Karin/MRI_Data/Controls/304/304_fMRI_preT_Anonymised/304_fMRI_preT.nii.gz → /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-304/ses-1/func/sub-304_ses-1_task-digitorder_bold.nii.gz
Copied /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/from_Karin/MRI_Data/Controls/308/308_3DT1_FSPGR_172_Anonymised/308_3DT1.nii → /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-308/ses-1/anat/sub-308_ses-1_T1w.nii.gz
Copied /mn

In [3]:
import os
import json
from glob import glob

bids_root = bids_folder
repetition_time = 2.1  # <-- Adjust if your TR is different

for root, dirs, files in os.walk(bids_root):
    for fname in files:
        if fname.endswith('_bold.nii.gz'):
            json_path = os.path.join(root, fname.replace('.nii.gz', '.json'))
            data = {
                "RepetitionTime": repetition_time,
                "TaskName": "digitorder"
            }
            with open(json_path, 'w') as f:
                json.dump(data, f, indent=4)
            print(f"Created: {json_path}")
        
        elif fname.endswith('_T1w.nii.gz'):
            json_path = os.path.join(root, fname.replace('.nii.gz', '.json'))
            data = {}
            with open(json_path, 'w') as f:
                json.dump(data, f, indent=4)
            print(f"Created: {json_path}")


Created: /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-115/ses-1/anat/sub-115_ses-1_T1w.json
Created: /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-115/ses-1/func/sub-115_ses-1_task-digitorder_bold.json
Created: /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-115/ses-2/func/sub-115_ses-2_task-digitorder_bold.json
Created: /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-123/ses-1/anat/sub-123_ses-1_T1w.json
Created: /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-123/ses-2/func/sub-123_ses-2_task-digitorder_bold.json
Created: /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-124/ses-1/anat/sub-124_ses-1_T1w.json
Created: /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-124/ses-1/func/sub-124_ses-1_task-digitorder_bold.json
Created: /mnt_AdaBD_largefiles/Data/SMILE_Data/DNumRisk/ds-numberline/sub-124/ses-2/func/sub-124_ses-2_task-digitorder_bold.json
Created: /mnt_AdaBD

### Problems with .nii.gz File type

error from fmriprep:
nibabel.filebasedimages.ImageFileError: Cannot work out file type of "/data/sub-115/ses-1/anat/sub-115_ses-1_T1w.nii.gz"

In [ ]:
# check 
sub = '133'
nifit_fn = op.join(bids_folder, f'sub-{sub}/ses-1/anat/sub-{sub}_ses-1_T1w.nii.gz')

import nibabel as nib
nifti_img = nib.load(nifit_fn)
nifti_img

In [8]:
import os
import gzip
import shutil
from pathlib import Path

# Set this to your BIDS root directory
bids_root = Path(bids_folder)

# Loop through all .nii.gz files in the dataset
for nii_gz_path in bids_root.rglob("*.nii.gz"):
    try:
        # Try opening with gzip to see if it's actually gzipped
        with gzip.open(nii_gz_path, "rb") as f:
            f.read(1)  # try reading one byte
        print(f"✅ Valid gzip: {nii_gz_path.name}")
    except (OSError, gzip.BadGzipFile):
        print(f"⚠️ Not a real .nii.gz file, fixing: {nii_gz_path.name}")

        # Step 1: Rename to .nii
        nii_path = nii_gz_path.with_suffix("")  # remove .gz suffix
        nii_gz_path.rename(nii_path)

        # Step 2: Recompress properly
        fixed_path = nii_path.with_suffix(".nii.gz")
        with open(nii_path, 'rb') as f_in, gzip.open(fixed_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)

        # Step 3: Remove original uncompressed .nii
        nii_path.unlink()

        print(f"✅ Recompressed correctly: {fixed_path.name}")


⚠️ Not a real .nii.gz file, fixing: sub-115_ses-1_T1w.nii.gz
✅ Recompressed correctly: sub-115_ses-1_T1w.nii.gz
✅ Valid gzip: sub-115_ses-1_task-digitorder_bold.nii.gz
✅ Valid gzip: sub-115_ses-2_task-digitorder_bold.nii.gz
⚠️ Not a real .nii.gz file, fixing: sub-123_ses-1_T1w.nii.gz
✅ Recompressed correctly: sub-123_ses-1_T1w.nii.gz
✅ Valid gzip: sub-123_ses-2_task-digitorder_bold.nii.gz
⚠️ Not a real .nii.gz file, fixing: sub-124_ses-1_T1w.nii.gz
✅ Recompressed correctly: sub-124_ses-1_T1w.nii.gz
✅ Valid gzip: sub-124_ses-1_task-digitorder_bold.nii.gz
✅ Valid gzip: sub-124_ses-2_task-digitorder_bold.nii.gz
⚠️ Not a real .nii.gz file, fixing: sub-126_ses-1_T1w.nii.gz
✅ Recompressed correctly: sub-126_ses-1_T1w.nii.gz
✅ Valid gzip: sub-126_ses-1_task-digitorder_bold.nii.gz
✅ Valid gzip: sub-126_ses-2_task-digitorder_bold.nii.gz
⚠️ Not a real .nii.gz file, fixing: sub-127_ses-1_T1w.nii.gz
✅ Recompressed correctly: sub-127_ses-1_T1w.nii.gz
✅ Valid gzip: sub-127_ses-1_task-digitorder_bold